<a href="https://colab.research.google.com/github/atebit4/ToyDecoder/blob/main/Decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Title: Decoder
By Ori for COSC462101



In [ ]:
#imports
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("Device:", device)

#starting value implementation
#corpus needs to be long enough to make training and validation examples at context length
#64 because 64/4 = 16

context_length = 16 #num of tokens model see
batch_size = 16
embedding_dim = 64  # vector size to each token
num_heads = 4       # number of attention heads
num_blocks = 1      # number of transform blocks
feedforward_dim = 256
#head_dimension = embedding_dim/num_heads... since num_heads = 1 then = embedding_dim



PyTorch version: 2.11.0+cpu
Device: cpu


In [ ]:
#TASK 1 PREPARE DATA
#sepeate words and punctuation

text = """
language models learn patterns from text data
attention helps a model connect related words
transformers process tokens with self attention
causal masking prevents a model from seeing future tokens
decoder models predict the next token in a sequence
token embeddings represent words as dense vectors
positional embeddings tell the model where tokens appear
the model reads previous tokens before predicting a new token
training data influences what a language model can learn
small datasets can cause repetitive generated text
attention weights show which tokens are relevant
multi head attention can learn different relationships
residual connections help information flow through a network
layer normalization stabilizes transformer training
feed forward layers transform each token representation
the decoder uses a causal mask during language modeling
the model predicts one token after another
better data can improve generated text quality
large language models require substantial computing resources
transformers are useful for many language tasks
students study how decoder only transformers generate text
the simplified model learns from a small classroom corpus
autoregressive generation uses earlier tokens as context
the next token depends on the tokens that came before it
causal attention makes generation possible without future information
"""

#seperate words for punct
def tokenize(text):
   return re.findall(r'\w+|[^\w\s]', text.lower())

tokens = tokenize(text)

#split 80 20 for training and validation
split = int(0.8 * len(tokens))
training_tokens, val_tokens = tokens[:split], tokens[split:]

#make vocabulary using training ONLY
vocabulary = ["<UNK>"]+sorted(set(training_tokens))
vocabulary_size = len(vocabulary)


string_to_index = {
    token: i for i, token in enumerate(vocabulary)
}
index_to_string = {
    i: token for token, i in string_to_index.items()
}

def encode(tokens):
    return [string_to_index.get(token, 0) for token in tokens]

def decode(indices):
    return [index_to_string[index] for index in indices]

#tokens to ints
training_id = encode(training_tokens)
val_id = encode(val_tokens)
#calc unk percentage
unknown_summation = sum(token_id == 0 for token_id in val_id)
unknown_percentage = 100 * unknown_summation / len(val_id)




#create input and target seq for next token prediction
seq_length= 8
batch = 8

def makeASequence(token_id, seq_length):
    input_seq = []
    target_seq = []
    for i in range(len(token_id)-seq_length):
      x = token_id[i:i+seq_length]
      y = token_id[i+1:i+seq_length+1]
      input_seq.append(x)
      target_seq.append(y)
    return torch.tensor(input_seq, dtype=torch.long), torch.tensor(target_seq, dtype=torch.long)

X_train, y_train = makeASequence(training_id, seq_length)
X_val, y_val = makeASequence(val_id, seq_length)

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_dataloader = DataLoader(train_dataset, batch_size=batch, shuffle=True)
val_loader= DataLoader(val_dataset, batch_size=batch, shuffle=False)




X_batch, y_batch = next(iter(train_dataloader))
print(X_batch.shape, y_batch.shape)



#print to consoles --------------------------
print("Vocabulary size:", vocabulary_size)
#print sample mapping
print("Sample mapping: ", list(string_to_index.items())[:10])
#print training id and training then length of val id and validation
print("Training id: ", training_id[:10])
print("Training length: ", len(training_id))
print("Validation id: ", val_id[:10])
print("Validation length: ", len(val_id))
print(f"Unknown percentage: {unknown_percentage:.2f}%")
print("Training example", len(train_dataset))
print("Validation example", len(val_dataset))
print("input:", decode(X_train[0].tolist()))
print("target:", decode(y_train[0].tolist()))


#validation gets high <UNK> because those words dont appear in the training portion
#because its such a small corpus it creates a difficult validation set


torch.Size([8, 8]) torch.Size([8, 8])
Vocabulary size: 104
Sample mapping:  [('<UNK>', 0), ('a', 1), ('after', 2), ('another', 3), ('appear', 4), ('are', 5), ('as', 6), ('attention', 7), ('before', 8), ('better', 9)]
Training id:  [39, 49, 43, 56, 28, 86, 16, 7, 33, 1]
Training length:  154
Validation id:  [0, 94, 0, 86, 87, 0, 47, 0, 28, 1]
Validation length:  39
Unknown percentage: 48.72%
Training example 146
Validation example 31
input: ['language', 'models', 'learn', 'patterns', 'from', 'text', 'data', 'attention']
target: ['models', 'learn', 'patterns', 'from', 'text', 'data', 'attention', 'helps']
